In [1]:
import random
filename = "D:/Desktop/KGCN-colakg-semantic/data/ml-1m/train.txt"

# user_id item_id1 item_id2 ...
user_items = {}
max_len = 0
cnt = 0
with open(filename, 'r') as file:
    for line in file:
        parts = line.strip().split()
        user_id = int(parts[0])
        item_ids = list(map(int, parts[1:]))
        max_len = max(max_len, len(item_ids))

        # item_id超过150个的，随机算去150个，并记录被截断人数cnt
        if len(item_ids) > 150:
            cnt+=1
            print(user_id)
            item_ids = random.sample(item_ids, 150)
        
        user_items[user_id] = item_ids


4
9
14
16
17
18
21
22
25
32
34
35
41
43
44
47
52
57
58
61
72
89
91
92
116
117
122
130
135
136
138
145
147
148
149
150
156
160
162
165
168
172
174
180
186
191
192
194
197
198
201
203
215
222
223
224
228
234
237
241
244
254
260
263
267
270
271
277
283
292
294
299
300
301
302
305
307
309
313
318
320
325
326
328
330
332
336
337
342
345
348
351
354
365
367
385
389
391
397
401
402
408
410
411
414
423
424
425
428
437
441
444
450
452
453
456
460
465
473
474
475
476
480
481
493
498
508
515
517
519
523
527
530
532
535
540
542
545
548
549
550
555
557
562
565
569
586
587
590
600
603
620
623
628
630
636
638
645
646
650
654
659
668
672
675
676
677
686
691
694
695
697
698
701
709
711
712
713
715
719
720
726
730
734
736
743
745
748
751
752
756
764
769
773
776
779
780
787
790
795
797
799
800
801
807
815
816
821
823
838
839
845
849
853
854
857
868
876
880
888
889
890
896
898
903
910
918
921
923
926
927
928
933
934
936
947
948
952
954
956
962
969
970
972
974
976
980
983
995
998
1000
1003
1009
1014
1016
1

In [2]:
user_items

{0: [32,
  41,
  18,
  20,
  36,
  43,
  35,
  25,
  7,
  4,
  24,
  49,
  0,
  16,
  12,
  8,
  2,
  3,
  31,
  50,
  33,
  14,
  1,
  27,
  45,
  17,
  19,
  38,
  40,
  21,
  22,
  6,
  47,
  46,
  9,
  37,
  44,
  13,
  15,
  48,
  5,
  51,
  10],
 1: [105,
  135,
  160,
  75,
  159,
  133,
  151,
  110,
  123,
  88,
  68,
  150,
  167,
  126,
  87,
  93,
  138,
  157,
  71,
  18,
  55,
  162,
  0,
  132,
  20,
  137,
  104,
  79,
  169,
  158,
  125,
  114,
  100,
  136,
  106,
  59,
  144,
  57,
  127,
  76,
  86,
  62,
  140,
  53,
  111,
  78,
  145,
  148,
  85,
  146,
  74,
  80,
  91,
  73,
  149,
  143,
  166,
  94,
  116,
  42,
  54,
  99,
  90,
  134,
  98,
  139,
  124,
  67,
  128,
  61,
  84,
  102,
  103,
  118,
  174,
  117,
  56,
  47,
  92,
  77,
  63,
  107,
  95,
  173,
  81,
  155,
  165,
  66,
  109,
  121,
  83,
  97,
  60,
  70,
  171,
  163,
  58,
  131,
  115,
  119,
  120,
  152,
  52,
  82],
 2: [204,
  64,
  181,
  5,
  175,
  58,
  198,
  166,
  124,
  

In [3]:
cnt

1686

In [5]:
import pandas as pd
path = "D:/Desktop/KGCN-colakg-semantic/data/ml-1m"
movies = pd.read_csv(path + "/ml1m_extended_movie.csv")

movies['genres'] = movies['genres'].combine_first(movies['Genres'])  
# Convert the 'Date' column to datetime format
movies['release_date'] = pd.to_datetime(movies['release_date'])
# Convert the 'Date' column to year-month format
movies['release_date'] = movies['release_date'].dt.to_period('M')
movies['release_date'] = movies['release_date'].combine_first(movies['Year']) 
movies['director'] = movies['director'].fillna("unknown")
movies['actors'] = movies['actors'].fillna("unknown")
movies['overview'] = movies['overview'].fillna("unknown")
movies['writer'] = movies['writer'].fillna("unknown")

In [6]:
movies.columns

Index(['MovieID', 'Title', 'Genres', 'Year', 'imdb_id', 'id',
       'production_countries', 'genres', 'original_language', 'original_title',
       'title', 'overview', 'release_date', 'vote_average', 'director',
       'writer', 'actors'],
      dtype='object')

In [7]:
item_list = "D:/Desktop/KGCN-colakg-semantic/data/ml-1m/item_map.txt"
item_map = {}


with open(item_list, 'r') as file:
    for line in file:
        key, value = line.strip().split()
        item_map[key] = int(value)
movies["item_id"] = movies["MovieID"].apply(lambda x: item_map.get(str(x)))

movies = movies[["MovieID",'item_id', 'Title','Genres', 'director', 'actors']]

In [8]:
movies.columns ### director和actors信息获取源？

Index(['MovieID', 'item_id', 'Title', 'Genres', 'director', 'actors'], dtype='object')

In [9]:

user_text_dic = {}

for index, row in movies.iterrows():
    item_id = row['item_id']
    name = row['Title']
    genres = row['Genres']
    director = row['director']
    actors = row['actors']
    
    text_kg = f'{name}: {{"genres": {genres}, "director": "{director}", "main actors": "{actors}"}}'
    
    user_text_dic[item_id] = text_kg


user_text = {}
max_len = 0
for user, item_list in user_items.items():
    max_len = max(max_len, len(item_list))
    text_list = [user_text_dic[item] for item in item_list]
    result_string = "; ".join(text_list)
    user_text[user] = result_string

# {title}: {{"genres": {Genres}, "director": "{director}", "main actors": "{actors}"}}

In [10]:
import json

def prompt_generation(id, text):
   
    analysis = \
    f"""
    Assuming you're a film expert with access to a viewer's movie-watching history, where each entry is formatted as "movie_name: genres: xx, director: xx, main actors: xx)".
    {text}
    Please analyze and summarize this user's viewing preferences from the aspects of movie genres, directors, and actors. Your response should be a coherent and fluent paragraph, not exceeding 100 words.
    Here is a sample output:"This user has a strong preference for action-packed films, often combined with elements of sci-fi, drama, and adventure. They frequently watch movies directed by Steven Spielberg and Ridley Scott, indicating a taste for high-quality, visually compelling storytelling. The user also enjoys films featuring iconic actors such as Harrison Ford, Arnold Schwarzenegger, and Paul Newman. Their viewing history includes classics like "Star Wars," "Jurassic Park," and "The Terminator," showcasing a penchant for both timeless blockbusters and genre-defining cinema. Overall, their tastes lean towards thrilling, well-crafted narratives with strong directorial vision and memorable performances."
    """
    
    return analysis


question_dic = {}
for id, text in user_text.items():
    prompt = prompt_generation(id, text)        
    question_dic[id] = prompt


print("The number of requests is: ", len(list(question_dic.keys())))

with open('D:/Desktop/KGCN-colakg-semantic/data/ml-1m/llm_input_user.json', 'w') as f:
    json.dump(question_dic, f)

The number of requests is:  6040


In [ ]:
llm_input_user.json
{"0": 
"\n    Assuming you're a film expert with access to a viewer's movie-watching history, where each entry is formatted as 
\"movie_name: genres: xx, director: xx, main actors: xx)\".
\n    
One Flew Over the Cuckoo's Nest (1975): 
{\"genres\": Drama, \"director\": \"Milo\u0161 Forman\", 
\"main actors\": \"Jack Nicholson|Louise Fletcher|Danny DeVito|William Redfield\"}; 
...
}
\n    Please analyze and summarize this user's viewing preferences from the aspects of movie genres, directors, and actors. Your response should be a coherent and fluent paragraph, not exceeding 100 words.
\n    Here is a sample output:
\"This user has a strong preference for action-packed films, often combined with elements of sci-fi, drama, and adventure. 
They frequently watch movies directed by Steven Spielberg and Ridley Scott, 
indicating a taste for high-quality, visually compelling storytelling. 
The user also enjoys films featuring iconic actors such as Harrison Ford, 
Arnold Schwarzenegger, and Paul Newman. 
Their viewing history includes classics like \"Star Wars,\" \"Jurassic Park,\" and \"The Terminator,\" 
showcasing a penchant for both timeless blockbusters and genre-defining cinema. 
Overall, their tastes lean towards thrilling, well-crafted narratives with strong directorial vision 
and memorable performances.\"\n    ", 

In [ ]:
# llm_input_item.json
# {"34": "The movie is titled Toy Story (1995). 
# \n    
# \n    - The basic information from its first-degree neighbors in the movie knowledge graph includes:  
# Toy Story (1995), directed by John Lasseter, is an English-language film released in 1995-10. 
# The movie falls under the genres of Animation, Comedy, Family. 
# The main cast of this movie includes Tom Hanks, Tim Allen, Don Rickles, Jim Varney. 

# \n    - In addition, the sampled second-order information from the knowledge graph is: 
# Movies in the same/similar genres as Toy Story (1995) also include: 
# < Pete's Dragon (1977), Cats Don't Dance (1997), The Mighty Ducks (1992), The Prince of Egypt (1998), 
# A Close Shave (1995), Santa with Muscles (1996), The Wrong Trousers (1993), Hairspray (1988), 
# It Takes Two (1995), Pete's Dragon (1977) > . 

# The director of Toy Story (1995), John Lasseter, also directed 
# A Bug's Life (1998), Toy Story 2 (1999). 

# The lead actor of this movie, Tom Hanks, also starred in 
# Sleepless in Seattle (1993), Apollo 13 (1995), The Green Mile (1999), A League of Their Own (1992), 
# Volunteers (1985), Nothing in Common (1986), You've Got Mail (1998), Bachelor Party (1984), 
# The Money Pit (1986), Philadelphia (1993). 

# The lead actor of this movie, Tim Allen, also starred in The Santa Clause (1994), 
# Jungle2Jungle (a.k.a. Jungle 2 Jungle) (1997), For Richer or Poorer (1997), Toy Story 2 (1999), 
# Galaxy Quest (1999). The lead actor of this movie, Don Rickles, also starred in Kelly's Heroes (1970).'
# \n\n    ", 
